In [ ]:
import yaml
import os
import re

VERSION_FIELDS = {"firmwareVersion", "hardwareTargets"}

REQUIRED = {
  "Behavior": {
    "device": "Behavior",
    "whoAmI": 1216,
    "firmwareVersion": "3.2",
    "hardwareTargets": "1.1"
  },
  "ClockGenerator": {
    "device": "WhiteRabbit",
    "whoAmI": 1404,
    "firmwareVersion": "0.1",
    "hardwareTargets": "0.1"
  },
  "EnvironmentSensor": {
    "device": "EnvironmentSensor",
    "whoAmI": 1405,
    "firmwareVersion": "0.3",
    "hardwareTargets": "0.1"
  },
  "Lickometer": {
    "device": "LicketySplit",
    "whoAmI": 1400,
    "firmwareVersion": "0.0",
    "hardwareTargets": "0.5"
  },
  "Olfactometer": {
    "device": "Olfactometer",
    "whoAmI": 1140,
    "firmwareVersion": "1.4",
    "hardwareTargets": "1.0 | 1.1"
  },
  "SniffDetector": {
    "device": "SniffDetector",
    "whoAmI": 1401,
    "firmwareVersion": "0.1",
    "hardwareTargets": "0.1"
  },
  "StepperDriver": {
    "device": "StepperDriver",
    "whoAmI": 1130,
    "firmwareVersion": "0.7",
    "hardwareTargets": "1.0"
  },
  "Treadmill": {
    "device": "Treadmill",
    "whoAmI": 1402,
    "firmwareVersion": "0.0",
    "hardwareTargets": "0.0"
  }
}

def device_versions(base_path: str) -> dict[str, dict[str, str]]:
    """
    Reads the session folder and returns a dictionary of all HARP devices
    and their versions.

    Each .harp folder is expected to contain a device.yml file with fields:
        device:
        whoAmI:
        firmwareVersion:
        hardwareTargets:

    Returns
    -------
    dict
        {
            "(device).harp": {
                "device": "...",
                "whoAmI": "...",
                "firmwareVersion": "...",
                "hardwareTargets": "..."
            },
            ...
        }
    """
    devices = {}
    behavior_path = os.path.join(base_path, "behavior")
    if not os.path.isdir(behavior_path):
        raise FileNotFoundError(f"Behavior folder not found: {behavior_path}")

    for folder in os.listdir(behavior_path):
        if not folder.endswith(".harp"):
            continue

        yml_path = os.path.join(behavior_path, folder, "device.yml")

        if not os.path.isfile(yml_path):
            # Skip devices without metadata
            continue

        try:
            with open(yml_path, "r") as f:
                yml = yaml.safe_load(f)
        except Exception as e:
            print(f"Could not read {yml_path}: {e}")
            continue

        devices[folder.replace(".harp", "")] = {
            "device": yml.get("device"),
            "whoAmI": yml.get("whoAmI"),
            "firmwareVersion": yml.get("firmwareVersion"),
            "hardwareTargets": yml.get("hardwareTargets"),
        }
    return devices

def _is_version_field(field_name: str) -> bool:
    return field_name in VERSION_FIELDS

def _parse_version(version: str) -> tuple[int, ...]|None:
    """
    Parse semantic-like version into tuple of ints.
    "1.2.3" -> (1, 2, 3)
    """
    try:
        return tuple(int(x) for x in version.strip().split("."))
    except ValueError:
        return None

def _compare_versions(v1: tuple[int, ...], v2: tuple[int, ...]) -> int:
    """
    Compare two version tuples.
    Returns:
        -1 if v1 < v2
         0 if equal
         1 if v1 > v2
    """
    max_len = max(len(v1), len(v2))
    v1 += (0,) * (max_len - len(v1))
    v2 += (0,) * (max_len - len(v2))

    if v1 < v2:
        return -1
    if v1 > v2:
        return 1
    return 0

def _check_constraint(current: str, constraint: str) -> bool:
    """
    Evaluate single constraint expression like:
    >=1.2, <2.0, ==1.3, !=1.0
    """
    match = re.match(r"(>=|<=|>|<|==|!=)?\s*([\d.]+)", constraint.strip())
    if not match:
        return False

    operator, required_version = match.groups()
    operator = operator or "=="

    current_parsed = _parse_version(current)
    required_parsed = _parse_version(required_version)

    if current_parsed is None or required_parsed is None:
        return False

    cmp_result = _compare_versions(current_parsed, required_parsed)

    return {
        "==": cmp_result == 0,
        "!=": cmp_result != 0,
        ">":  cmp_result == 1,
        "<":  cmp_result == -1,
        ">=": cmp_result in (0, 1),
        "<=": cmp_result in (0, -1),
    }[operator]

def _version_matches(current, requirement) -> bool:
    """
    Supports:
    - plain versions ("1.2")
    - numeric upgrade allowed
    - boolean true/any
    - constraint expressions (>=1.2, <2.0, etc.)
    - unions using | or ||
    """

    current_str = str(current).strip()
    req_str = str(requirement).strip()

    # Boolean support
    if req_str.lower() in {"true", "any"}:
        return True
    if req_str.lower() == "false":
        return False

    # Union support
    if "|" in req_str:
        parts = re.split(r"\s*\|\|\s*|\s*\|\s*", req_str)
        return any(_check_constraint(current_str, part) for part in parts)

    # Constraint support
    if re.match(r"(>=|<=|>|<|==|!=)", req_str):
        return _check_constraint(current_str, req_str)

    # Plain version: allow upgrade
    current_parsed = _parse_version(current_str)
    req_parsed = _parse_version(req_str)

    if current_parsed and req_parsed:
        return _compare_versions(current_parsed, req_parsed) >= 0

    return current_str == req_str

def check_devices_version(versions, required_versions, path="")-> None:
    for key in versions.keys() | required_versions.keys():
        current_path = f"{path}.{key}" if path else key

        if key not in versions:
            # print(f"[EXTRA] {current_path}: {required_versions[key]}")
            continue
        elif key not in required_versions:
            print(f"[Missing version to be compared to] {current_path}: {versions[key]}")
        else:
            ver = versions[key]
            req_ver = required_versions[key]

            if isinstance(ver, dict) and isinstance(req_ver, dict):
                check_devices_version(ver, req_ver, current_path)

            elif _is_version_field(key):
                if not _version_matches(ver, req_ver):
                    print(f"  [VERSION MISMATCH] {current_path}:")
                    print(f"    - Current: {ver}")
                    print(f"    - Required: {req_ver}")

            elif ver != req_ver:
                print(f"{current_path}:")
                print(f"    - Current: {ver}")
                print(f"    - Required: {req_ver}")


In [ ]:
base_path = r"C:/Data/tests/828425_2026-02-11T203733Z"

versions = device_versions(base_path)

# print(f"Device versions: {json.dumps(device_versions(base_path), indent=2)}")

check_devices_version(versions, REQUIRED)


In [ ]:
from utils.data_loading import load_rig_info

base_root = r"C:/Data/tests/"

for rig_folder in os.listdir(base_root):
    rig_path = os.path.join(base_root, rig_folder)
    if not os.path.isdir(rig_path):
        continue

    for subfolder in os.listdir(rig_path):
        full_path = os.path.join(rig_path, subfolder)
        if not os.path.isdir(full_path):
            continue

        rig_info = load_rig_info(rig_path)
        versions = device_versions(base_path)
        check_devices_version(versions, REQUIRED)
